# Attention, step by step

This notebook builds **causal self-attention** from nothing, on a toy example small enough
to read every number. Each step prints its tensors. By the end we will have re-created,
line by line, the `CausalSelfAttentionHead` class in `mini_gpt/model.py`, and we will check
that our hand-made version gives exactly the same answer.

**How to use it:** run the cells top to bottom (Shift+Enter). Change numbers and re-run;
breaking things on purpose is the fastest way to understand them.

Shape letters used throughout: `T` = number of tokens in the sequence, `C` = size of each
token vector, `B` = batch size (how many sequences at once).

In [1]:
import math
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

# Make `import mini_gpt` work whether the notebook is run from notebooks/ or the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

torch.manual_seed(42)                                   # same "random" numbers every run
torch.set_printoptions(precision=3, sci_mode=False)     # short, readable numbers
print("PyTorch", torch.__version__)

PyTorch 2.14.1+cu130


## 1. From text to vectors

A model cannot read letters, so we go through two steps:

1. **Tokenize**: give every distinct character an integer ID.
2. **Embed**: look up a (learnable) vector for each ID.

In [2]:
text = "hello"
chars = sorted(set(text))                 # the vocabulary: every distinct character
stoi = {ch: i for i, ch in enumerate(chars)}
ids = torch.tensor([stoi[ch] for ch in text])

print("vocabulary:", chars)
print("token IDs :", ids.tolist(), " <- one integer per character of", repr(text))

vocabulary: ['e', 'h', 'l', 'o']
token IDs : [1, 0, 2, 2, 3]  <- one integer per character of 'hello'


In [3]:
T = len(ids)   # 5 tokens
C = 4          # each token becomes a vector of 4 numbers

token_emb = nn.Embedding(num_embeddings=len(chars), embedding_dim=C)
x = token_emb(ids).detach()   # (T, C). detach() = "just numbers, no gradient tracking" for this demo

print("embedding table (one row per vocabulary entry):")
print(token_emb.weight.detach())
print("\nx = one row per token of 'hello', shape", tuple(x.shape))
print(x)

embedding table (one row per vocabulary entry):
tensor([[ 1.927,  1.487,  0.901, -2.106],
        [ 0.678, -1.235, -0.043, -1.605],
        [-0.752,  1.649, -0.392, -1.404],
        [-0.728, -0.559, -0.769,  0.762]])

x = one row per token of 'hello', shape (5, 4)
tensor([[ 0.678, -1.235, -0.043, -1.605],
        [ 1.927,  1.487,  0.901, -2.106],
        [-0.752,  1.649, -0.392, -1.404],
        [-0.752,  1.649, -0.392, -1.404],
        [-0.728, -0.559, -0.769,  0.762]])


Notice that the two `l` tokens (rows 2 and 3) have **identical** vectors: an embedding only
knows *which* character it is, not *where* it is. We will come back to that in section 8.

## 2. The goal: let tokens talk to earlier tokens

Right now each row of `x` knows only about itself. To predict the next character, a token
needs information from the tokens *before* it. The simplest way to "gather information"
is to **average**: replace each token's vector by the mean of itself and all earlier tokens.

### Version 1: a plain loop

In [4]:
avg_loop = torch.zeros(T, C)
for t in range(T):
    earlier = x[: t + 1]              # tokens 0..t  (never t+1 or later: that would be cheating)
    avg_loop[t] = earlier.mean(dim=0)

print(avg_loop)
print("\nrow 0 equals x[0] (nothing earlier to average with):", torch.allclose(avg_loop[0], x[0]))

tensor([[ 0.678, -1.235, -0.043, -1.605],
        [ 1.303,  0.126,  0.429, -1.855],
        [ 0.618,  0.634,  0.155, -1.705],
        [ 0.275,  0.888,  0.018, -1.629],
        [ 0.075,  0.598, -0.139, -1.151]])

row 0 equals x[0] (nothing earlier to average with): True


### Version 2: the same thing as ONE matrix multiplication

Build a `T x T` matrix of weights where row `t` says *how much of each token goes into
position t*. For an average over tokens `0..t`, row `t` is `1/(t+1)` for the first `t+1`
entries and `0` afterwards. That is a lower-triangular matrix.

In [5]:
tril = torch.tril(torch.ones(T, T))               # 1s on and below the diagonal
weights = tril / tril.sum(dim=1, keepdim=True)    # divide each row by its sum
print("tril =")
print(tril)
print("\nweights (each row sums to 1) =")
print(weights)

avg_matrix = weights @ x                          # (T, T) @ (T, C) -> (T, C)
print("\nsame result as the loop:", torch.allclose(avg_matrix, avg_loop))

tril =
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

weights (each row sums to 1) =
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.500, 0.500, 0.000, 0.000, 0.000],
        [0.333, 0.333, 0.333, 0.000, 0.000],
        [0.250, 0.250, 0.250, 0.250, 0.000],
        [0.200, 0.200, 0.200, 0.200, 0.200]])

same result as the loop: True


### Version 3: get the same weights from a softmax

Start from scores of `0` everywhere, put `-inf` wherever a token must not look (the future),
and apply **softmax** to each row. Softmax computes `exp(score) / sum(exp(scores))`:
`exp(0) = 1` and `exp(-inf) = 0`, so the allowed positions share the weight equally.

This looks like a roundabout way to get an average, but it is the key step: in a moment
the zeros will be replaced by **learned, data-dependent scores**.

In [6]:
scores = torch.zeros(T, T)
scores = scores.masked_fill(tril == 0, float("-inf"))
print("scores =")
print(scores)

weights_softmax = F.softmax(scores, dim=-1)
print("\nsoftmax(scores) =")
print(weights_softmax)
print("\nsame weights as before:", torch.allclose(weights_softmax, weights))

scores =
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

softmax(scores) =
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.500, 0.500, 0.000, 0.000, 0.000],
        [0.333, 0.333, 0.333, 0.000, 0.000],
        [0.250, 0.250, 0.250, 0.250, 0.000],
        [0.200, 0.200, 0.200, 0.200, 0.200]])

same weights as before: True


## 3. Self-attention: let the data decide the weights

A plain average treats every earlier token as equally important. That is rarely right:
after `q` you care a lot about the `q` and very little about a space ten characters back.

Self-attention lets each token compute its own weights. Every token produces three vectors
using three learned matrices:

| vector | question it answers | formula |
|---|---|---|
| **query** `q` | "what am I looking for?" | `q = x @ W_q` |
| **key** `k` | "what do I contain?" | `k = x @ W_k` |
| **value** `v` | "what will I hand over if someone picks me?" | `v = x @ W_v` |

The score between token `i` and token `j` is the **dot product** `q_i . k_j`: large when
the query of `i` and the key of `j` point in the same direction.

In [7]:
head_size = 4
torch.manual_seed(0)
query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

with torch.no_grad():
    q = query(x)   # (T, head_size)
    k = key(x)     # (T, head_size)
    v = value(x)   # (T, head_size)

print("q ="); print(q)
print("\nk ="); print(k)

q =
tensor([[ 0.275, -0.932, -0.029,  0.064],
        [ 0.796, -1.015,  0.182, -1.638],
        [ 1.123, -0.187,  0.449, -0.132],
        [ 1.123, -0.187,  0.449, -0.132],
        [-0.111,  0.375, -0.000,  0.706]])

k =
tensor([[ 0.128, -0.985,  0.605, -0.742],
        [ 0.980,  0.087,  1.308, -1.880],
        [ 0.785,  0.063,  0.621, -0.502],
        [ 0.785,  0.063,  0.621, -0.502],
        [-0.217, -0.000, -0.673,  0.779]])


In [8]:
# Every query dotted with every key: a (T, T) table of raw scores.
raw_scores = q @ k.T
print("raw scores  [row i, column j] = q_i . k_j")
print(raw_scores)

# Check one entry by hand: token 4 ('o') looking at token 1 ('e').
by_hand = sum(q[4, d] * k[1, d] for d in range(head_size))
print(f"\nq[4] . k[1] by hand = {by_hand:.3f}   table entry = {raw_scores[4, 1]:.3f}")

raw scores  [row i, column j] = q_i . k_j
tensor([[ 0.888,  0.030,  0.106,  0.106,  0.010],
        [ 2.427,  4.008,  1.495,  1.495, -1.571],
        [ 0.697,  1.919,  1.214,  1.214, -0.648],
        [ 0.697,  1.919,  1.214,  1.214, -0.648],
        [-0.908, -1.404, -0.418, -0.418,  0.574]])

q[4] . k[1] by hand = -1.404   table entry = -1.404


### Scale, mask, softmax

Three small steps turn raw scores into weights:

1. **Scale**: divide by `sqrt(head_size)` (section 6 shows why).
2. **Mask**: put `-inf` above the diagonal so no token can look at the future.
3. **Softmax**: turn each row into positive weights that sum to 1.

In [9]:
scores = raw_scores / math.sqrt(head_size)
scores = scores.masked_fill(tril == 0, float("-inf"))
print("scaled + masked scores =")
print(scores)

att = F.softmax(scores, dim=-1)
print("\nattention weights =")
print(att)
print("\nrow sums:", att.sum(dim=-1))

scaled + masked scores =
tensor([[ 0.444,   -inf,   -inf,   -inf,   -inf],
        [ 1.214,  2.004,   -inf,   -inf,   -inf],
        [ 0.348,  0.960,  0.607,   -inf,   -inf],
        [ 0.348,  0.960,  0.607,  0.607,   -inf],
        [-0.454, -0.702, -0.209, -0.209,  0.287]])

attention weights =
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.312, 0.688, 0.000, 0.000, 0.000],
        [0.242, 0.445, 0.313, 0.000, 0.000],
        [0.184, 0.339, 0.238, 0.238, 0.000],
        [0.155, 0.121, 0.199, 0.199, 0.326]])

row sums: tensor([1.000, 1.000, 1.000, 1.000, 1.000])


Read the last row: it tells you how much the final token `o` draws from each of
`h, e, l, l, o`. The numbers are no longer a flat `0.2` each: the (random, untrained)
`W_q` and `W_k` already prefer some tokens over others. Training adjusts those matrices so
that the preferences become *useful*.

Rows 2 and 3 belong to the two identical `l` tokens. Their raw scores against tokens 0-2
are the same; the weights differ only because row 3 is allowed to look at one more token.

### The output: a weighted average of the values

In [10]:
out = att @ v     # (T, T) @ (T, head_size) -> (T, head_size)
print("out =")
print(out)

# Row 4 by hand: sum over j of att[4, j] * v[j]
by_hand = sum(att[4, j] * v[j] for j in range(T))
print("\nrow 4 by hand:", by_hand)
print("row 4 of out :", out[4])

out =
tensor([[ 0.831, -1.127,  1.080, -0.411],
        [ 0.056, -0.512,  0.838, -1.100],
        [ 0.275, -0.219,  0.642, -0.944],
        [ 0.367, -0.048,  0.531, -0.883],
        [ 0.445, -0.009,  0.231, -0.382]])

row 4 by hand: tensor([ 0.445, -0.009,  0.231, -0.382])
row 4 of out : tensor([ 0.445, -0.009,  0.231, -0.382])


## 4. Checking causality

The whole point of the mask: the output at position `t` must depend only on tokens `0..t`.
Let us test it. We change the **last** token and check that the outputs at positions 0-3
do not change at all.

In [11]:
def attention(x):
    # Everything from section 3 in one function. x: (T, C) -> (T, head_size)
    with torch.no_grad():
        q, k, v = query(x), key(x), value(x)
    n = x.shape[0]
    mask = torch.tril(torch.ones(n, n))
    scores = (q @ k.T / math.sqrt(head_size)).masked_fill(mask == 0, float("-inf"))
    return F.softmax(scores, dim=-1) @ v

x_changed = x.clone()
x_changed[4] = torch.randn(C)          # replace the last token with something else

out_a, out_b = attention(x), attention(x_changed)
print("positions 0-3 unchanged:", torch.equal(out_a[:4], out_b[:4]))
print("position 4 changed     :", not torch.allclose(out_a[4], out_b[4]))

positions 0-3 unchanged: True
position 4 changed     : True


Why does this matter? During training we feed in a whole sequence and ask for a prediction
at **every** position at once (position `t` must predict token `t+1`). If position `t`
could see token `t+1`, it would simply copy the answer and learn nothing.

## 5. What happens without the mask?

In [12]:
def attention_no_mask(x):
    with torch.no_grad():
        q, k, v = query(x), key(x), value(x)
    return F.softmax(q @ k.T / math.sqrt(head_size), dim=-1) @ v

out_a, out_b = attention_no_mask(x), attention_no_mask(x_changed)
print("positions 0-3 unchanged:", torch.allclose(out_a[:4], out_b[:4]), " <- the future leaked into the past")

positions 0-3 unchanged: False  <- the future leaked into the past


## 6. Why divide by `sqrt(head_size)`?

A dot product adds up `head_size` terms, so its typical size grows like `sqrt(head_size)`.
Softmax is very sensitive to the size of its inputs: big numbers make it put almost all
the weight on one entry ("one-hot"), and then the gradient through it is nearly zero and
learning stalls. Dividing by `sqrt(head_size)` keeps the scores at a steady size no matter
how wide the head is.

In [13]:
torch.manual_seed(1)
for size in [4, 64, 1024]:
    a, b = torch.randn(1000, size), torch.randn(1000, size)
    dots = (a * b).sum(dim=1)                       # 1000 random dot products
    print(f"head_size {size:4d}:  spread of raw dot products = {dots.std():6.2f}"
          f"   after dividing by sqrt(head_size) = {(dots / math.sqrt(size)).std():.2f}")

small = torch.tensor([0.5, -0.3, 0.1, 0.8])
print("\nsoftmax of small scores:", F.softmax(small, dim=-1))
print("softmax of 16x larger  :", F.softmax(small * 16, dim=-1), " <- nearly one-hot")

head_size    4:  spread of raw dot products =   2.03   after dividing by sqrt(head_size) = 1.01
head_size   64:  spread of raw dot products =   7.77   after dividing by sqrt(head_size) = 0.97
head_size 1024:  spread of raw dot products =  31.68   after dividing by sqrt(head_size) = 0.99

softmax of small scores: tensor([0.288, 0.130, 0.193, 0.389])
softmax of 16x larger  : tensor([0.008, 0.000, 0.000, 0.992])  <- nearly one-hot


## 7. Our version vs the repo's `CausalSelfAttentionHead`

The class in `mini_gpt/model.py` does exactly the steps above (plus a batch dimension and
dropout). If we copy our three matrices into it, the outputs must match exactly.

In [14]:
from mini_gpt.model import CausalSelfAttentionHead, MultiHeadAttention

head = CausalSelfAttentionHead(n_embd=C, head_size=head_size, block_size=T)
head.query.load_state_dict(query.state_dict())
head.key.load_state_dict(key.state_dict())
head.value.load_state_dict(value.state_dict())

with torch.no_grad():
    class_out, class_att = head(x.unsqueeze(0))   # unsqueeze(0): (T, C) -> (1, T, C), a batch of one

print("output shape :", tuple(class_out.shape), "  weights shape:", tuple(class_att.shape))
print("outputs match:", torch.allclose(class_out[0], out))
print("weights match:", torch.allclose(class_att[0], att))

output shape : (1, 5, 4)   weights shape: (1, 5, 5)
outputs match: True
weights match: True


## 8. Attention cannot see order (so we add positions)

Look at what attention actually computes: a weighted **sum** over earlier tokens. A sum
does not care about the order of the things being added. So if we shuffle the earlier
tokens, the last token's output stays the same:

In [15]:
x_shuffled = x[[2, 0, 3, 1, 4]]          # reorder tokens 0-3, keep the last token in place
print("last-position output, original order:", attention(x)[4])
print("last-position output, shuffled order:", attention(x_shuffled)[4])

last-position output, original order: tensor([ 0.445, -0.009,  0.231, -0.382])
last-position output, shuffled order: tensor([ 0.445, -0.009,  0.231, -0.382])


To the model, `"hell" + "o"` and `"lhle" + "o"` look identical from the last position.
The fix is the **positional embedding**: a second lookup table with one learned vector
per *position*, added to the token vector.

In [16]:
pos_emb = nn.Embedding(num_embeddings=T, embedding_dim=C)
positions = torch.arange(T)                       # tensor([0, 1, 2, 3, 4])
x_pos = (token_emb(ids) + pos_emb(positions)).detach()

print("without positions, the two l's are identical:", torch.equal(x[2], x[3]))
print("with positions, they differ                 :", not torch.equal(x_pos[2], x_pos[3]))

shuffled_ids = ids[[2, 0, 3, 1, 4]]
x_pos_shuffled = (token_emb(shuffled_ids) + pos_emb(positions)).detach()
print("\nlast-position output, original order:", attention(x_pos)[4])
print("last-position output, shuffled order:", attention(x_pos_shuffled)[4], " <- now different")

without positions, the two l's are identical: True
with positions, they differ                 : True

last-position output, original order: tensor([ 0.294,  0.400, -0.274, -0.005])
last-position output, shuffled order: tensor([ 0.340,  0.357, -0.247,  0.020])  <- now different


## 9. Multi-head attention

One head produces one set of weights, so it can follow one kind of relationship at a time.
**Multi-head attention** runs several smaller heads side by side and concatenates their
outputs. With `C = 4` and 2 heads, each head has size `4 / 2 = 2`.

In [17]:
torch.manual_seed(0)
mha = MultiHeadAttention(n_embd=C, n_head=2, block_size=T)

with torch.no_grad():
    mha_out, mha_att = mha(x_pos.unsqueeze(0))

print("output shape :", tuple(mha_out.shape), "  (same as the input, so layers can be stacked)")
print("weights shape:", tuple(mha_att.shape), "  (batch, heads, T, T)")
print("\nhead 0 weights:"); print(mha_att[0, 0])
print("\nhead 1 weights:"); print(mha_att[0, 1])

output shape : (1, 5, 4)   (same as the input, so layers can be stacked)
weights shape: (1, 2, 5, 5)   (batch, heads, T, T)

head 0 weights:
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.221, 0.779, 0.000, 0.000, 0.000],
        [0.297, 0.318, 0.385, 0.000, 0.000],
        [0.175, 0.388, 0.195, 0.242, 0.000],
        [0.205, 0.049, 0.266, 0.164, 0.315]])

head 1 weights:
tensor([[1.000, 0.000, 0.000, 0.000, 0.000],
        [0.994, 0.006, 0.000, 0.000, 0.000],
        [0.272, 0.380, 0.349, 0.000, 0.000],
        [0.537, 0.097, 0.171, 0.195, 0.000],
        [0.011, 0.268, 0.096, 0.074, 0.551]])


The two heads have different weights: each one has its own `W_q`, `W_k`, `W_v`, so each
looks at the sequence in its own way.

## 10. The whole model on our toy sentence

Finally, the full `GPT` class: embeddings, a stack of blocks (attention + feed-forward,
each wrapped in a residual connection), and a final layer that outputs one score per
vocabulary entry. We give it `"hell"` and ask it to predict `"ello"`.

In [18]:
from mini_gpt.config import ModelConfig
from mini_gpt.model import GPT

torch.manual_seed(0)
cfg = ModelConfig(vocab_size=len(chars), block_size=8, n_layer=2, n_head=2, n_embd=16, dropout=0.0)
model = GPT(cfg)
print(f"parameters: {model.num_params():,}")

inputs  = ids[:-1].unsqueeze(0)   # "hell"  shape (1, 4)
targets = ids[1:].unsqueeze(0)    # "ello"  shape (1, 4): the same text shifted by one

logits, loss = model(inputs, targets)
print("logits shape:", tuple(logits.shape), " (batch, T, vocab_size)")
print(f"loss before training: {loss.item():.3f}   (uniform guessing = ln({len(chars)}) = {math.log(len(chars)):.3f})")

parameters: 6,416

logits shape: (1, 4, 4)  (batch, T, vocab_size)
loss before training: 1.431   (uniform guessing = ln(4) = 1.386)


In [19]:
# Train on this single example until the model has memorised it.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
for step in range(101):
    logits, loss = model(inputs, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

probs = F.softmax(logits[0], dim=-1).detach()
print("\nprobabilities for the next character at each position (columns:", chars, ")")
print(probs)
print("prediction:", "".join(chars[i] for i in probs.argmax(dim=-1)), " (target: 'ello')")

step   0  loss 1.4311


step  25  loss 0.0451
step  50  loss 0.0073


step  75  loss 0.0041
step 100  loss 0.0029

probabilities for the next character at each position (columns: ['e', 'h', 'l', 'o'] )
tensor([[0.996, 0.003, 0.001, 0.000],
        [0.001, 0.001, 0.998, 0.001],
        [0.001, 0.001, 0.998, 0.001],
        [0.000, 0.003, 0.001, 0.996]])
prediction: ello  (target: 'ello')


The model has learned `h -> e`, `he -> l`, `hel -> l`, `hell -> o`. Note the third and
fourth predictions: the input token is `l` both times, yet the model predicts `l` the first
time and `o` the second. A bigram model (which only sees the current character) could never
do that. Attention plus positions lets the model use the **context**.

## Where to go next

- Read `mini_gpt/model.py` top to bottom: every class there now corresponds to a section above.
- `docs/LEARNING_GUIDE.md` chapters 9-12 explain the same ideas in more depth, with exercises.
- Run `python scripts/visualize_attention.py` to see the attention weights of the *trained* model.